# Day 23 — Virtual Environments & Packages

**How to use this notebook:**
- Read each exercise, fill in your solution (replace `pass`)
- Run the test cell right after to see **APPROVED** or **FAILED**
- Do NOT modify the test cells

**Key concepts today:** virtual environments, `pip`, package organisation, `__init__.py`, `importlib`, `sys.modules`

> Day 23 is primarily about tooling. These exercises turn the concepts into testable Python code.

---
## Exercise 1 — Check Package Availability

Write a function that checks whether a package is available to import.

Return `True` if the package can be imported, `False` if not.

**Hint:** Use `importlib.util.find_spec(name)` — it returns `None` if the package is not found.

| Input | Expected Output |
|---|---|
| `'os'` | `True` |
| `'math'` | `True` |
| `'totally_fake_package_xyz'` | `False` |

In [3]:
import importlib.util

def is_package_available(name):
    if importlib.util.find_spec(name) is not None:
        return True
    else:
        return False

In [4]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 1: is_package_available ──')
_check('os',      is_package_available('os'),   True)
_check('math',    is_package_available('math'), True)
_check('fake_pkg',is_package_available('totally_fake_package_xyz'), False)


── Exercise 1: is_package_available ──
  ✔ APPROVED — os
  ✔ APPROVED — math
  ✔ APPROVED — fake_pkg


---
## Exercise 2 — Module Inspector

Write a function `module_info(name)` that:
1. Attempts to import the module using `importlib.import_module(name)`.
2. If the import fails, returns `None`.
3. If successful, returns a dict:
   - `'name'`: the module's `__name__`
   - `'has_file'`: `True` if the module has a `__file__` attribute that is not `None`, else `False`
   - `'doc_start'`: first 20 characters of `__doc__` if `__doc__` is not `None`, else `''`

| Input | Expected keys |
|---|---|
| `'os'`   | name='os', has_file=True, doc_start starts with 'OS routines' |
| `'math'` | name='math', has_file depends on platform |
| `'fake_xyz_module_99'` | `None` |

In [5]:
import importlib

def module_info(name):
    try:
        module = importlib.import_module(name)
        has_file = module.__file__ is not None

        if module.__doc__ is not None:
            doc_start = module.__doc__[:20]
        else:
            doc_start = ''
                
    except:
        return None
    
    return {
        'name': module.__name__, 
        'has_file': has_file,
        'doc_start': doc_start
    }

In [6]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Exercise 2: module_info ──')
info_os = module_info('os')
_check('os name',     info_os['name'],     'os')
_check('os has_file', info_os['has_file'], True)
_check('os doc_start starts with OS', info_os['doc_start'].startswith('OS routines'), True)

info_fake = module_info('fake_xyz_module_99')
_check('fake returns None', info_fake, None)

info_json = module_info('json')
_check('json name', info_json['name'], 'json')
_check('json doc_start type', isinstance(info_json['doc_start'], str), True)


── Exercise 2: module_info ──
  ✔ APPROVED — os name
  ✔ APPROVED — os has_file
  ✔ APPROVED — os doc_start starts with OS
  ✔ APPROVED — fake returns None
  ✔ APPROVED — json name
  ✔ APPROVED — json doc_start type


---
## Mini-Project — Package Manager Simulator

Simulate a lightweight package manager that tracks installed packages.

- `install(registry, name, version)` — adds `name: version` to registry dict. If already installed with same version → return `'Already installed'`. If different version → return `'Updated {name} to {version}'`. If new → return `'Installed {name}=={version}'`.
- `uninstall(registry, name)` — removes package. Returns `'Removed {name}'` or `'Package not found'`.
- `list_packages(registry)` — returns sorted list of `'{name}=={version}'` strings.
- `check_conflicts(registry, requirements)` — `requirements` is a list of `(name, min_version)` tuples. Returns list of packages where installed version < min_version (using string comparison is fine here).

In [11]:
def install(registry, name, version):
    if name in registry:
        if registry[name] == version:
            return 'Already installed'
        elif registry[name] != version:
            registry[name] = version
            return f'Updated {name} to {version}'
    else:
        registry[name] = version
        return f'Installed {name}=={version}'

def uninstall(registry, name):
    if name in registry:
        del registry[name]
        return f'Removed {name}'
    else:
        return 'Package not found'

def list_packages(registry):
    return sorted([f'{name}=={version}'for name, version in registry.items()])
    

def check_conflicts(registry, requirements):
    result = []
    for name, min_version in requirements:
        if registry[name] < min_version:
            results.append(name)
    return result

In [12]:
# TEST CELL — do not modify
def _check(label, got, expected):
    if got == expected:
        print(f'  ✔ APPROVED — {label}')
    else:
        print(f'  ✘ FAILED   — {label}')
        print(f'             got:      {got!r}')
        print(f'             expected: {expected!r}')

print('\n── Mini-Project: Package Manager ──')
reg = {}
_check('install flask',     install(reg,'flask','2.0.0'),   'Installed flask==2.0.0')
_check('install requests',  install(reg,'requests','2.1.0'),'Installed requests==2.1.0')
_check('already installed', install(reg,'flask','2.0.0'),   'Already installed')
_check('update flask',      install(reg,'flask','2.3.0'),   'Updated flask to 2.3.0')

_check('list packages', list_packages(reg), ['flask==2.3.0','requests==2.1.0'])

_check('uninstall requests',  uninstall(reg,'requests'), 'Removed requests')
_check('uninstall not found', uninstall(reg,'numpy'),    'Package not found')

reg2 = {'flask':'2.3.0','numpy':'1.20.0','pandas':'1.3.0'}
_check('conflicts', check_conflicts(reg2,[('flask','2.5.0'),('numpy','1.20.0'),('pandas','1.4.0')]),
       ['flask','pandas'])


── Mini-Project: Package Manager ──
  ✔ APPROVED — install flask
  ✔ APPROVED — install requests
  ✔ APPROVED — already installed
  ✔ APPROVED — update flask
  ✔ APPROVED — list packages
  ✔ APPROVED — uninstall requests
  ✔ APPROVED — uninstall not found


NameError: name 'results' is not defined